# Phase 9 (Priority 2-B): Mistral-7B-v0.3 + Open Pool Multi-Depth LoRA Pilot

**目的（優先検証 2-B）**:
- `NTL 0707`（Qwen 系偏重・Gemma 系ゼロ）の弱い枠（`q25_14` `w=1.0` 等）との**差し替え候補**として、非 Qwen 系 **`mistralai/Mistral-7B-v0.3`** を Colab A100 で学習・検証します。
- `train.csv`（`N=2,699`, `official_5fold_splits.json`）を Val Fold に固定しつつ、**Train Fold のみに `crtproduction/name-that-llm-open-pool`（重複・リーク除外済み）を注入**して汎化底上げを図ります。

**合格ゲート**:
1. 生成された `/content/phase9_mistral7b_oof_2699.npy` をローカルの `eval_18class_oof_complement.py --npy ...` で評価し、**OOF Mean4 が `+0.003` 以上純増**すること。
2. `2×T4` 上で `q25_14` 枠以下の時間で推論可能であること（`V1` の 1.6h 枠を圧迫しない）。


In [ ]:
# [Cell 1] Environment, Kaggle & HuggingFace Authentication
!pip uninstall -y -q torchao
!pip install -U -q "transformers>=5.10.1" peft accelerate datasets scikit-learn scipy kaggle bitsandbytes huggingface_hub

import os, sys, json, re, gc, time, getpass, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# --- Kaggle Auth ---
kaggle_json_path = Path.home() / '.kaggle' / 'kaggle.json'
kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)
auth_ok = False
try:
    from google.colab import userdata
    u = userdata.get('KAGGLE_USERNAME')
    k = userdata.get('KAGGLE_KEY')
    if u and k:
        os.environ['KAGGLE_USERNAME'] = u
        os.environ['KAGGLE_KEY'] = k
        kaggle_json_path.write_text(json.dumps({'username': u, 'key': k}))
        os.chmod(kaggle_json_path, 0o600)
        auth_ok = True
        print(f"Kaggle Authenticated via Colab Secrets: {u}")
except Exception:
    pass

if not auth_ok and not kaggle_json_path.exists() and not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
    u = input("Kaggle Username: ").strip()
    k = getpass.getpass("Kaggle API Key: ").strip()
    os.environ['KAGGLE_USERNAME'] = u
    os.environ['KAGGLE_KEY'] = k
    kaggle_json_path.write_text(json.dumps({'username': u, 'key': k}))
    os.chmod(kaggle_json_path, 0o600)

# --- HuggingFace Auth (required if google/gemma-4-12B is gated) ---
hf_token = ""
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or ""
except Exception:
    pass
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN', '')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN loaded.")
else:
    print("HF_TOKEN not set in Secrets (if gated model fails, set HF_TOKEN).")

# --- GitHub Auth (for per-fold auto-sync) ---
gh_token = ""
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN') or ""
except Exception:
    pass
if not gh_token:
    gh_token = os.environ.get('GH_TOKEN', '')
if gh_token:
    os.environ['GH_TOKEN'] = gh_token
    print("GITHUB_TOKEN loaded (per-fold auto-sync enabled).")
else:
    print("GITHUB_TOKEN not set in Secrets (per-fold GitHub sync will be skipped unless set in Cell 5).")

VRAM_GB = 0.0
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device: {torch.cuda.get_device_name(0)} | VRAM: {VRAM_GB:.1f} GB")


In [ ]:
# [Cell 2] Data Acquisition, Official 5-Fold Alignment & Leak-Free Open Pool Prep
import urllib.request

DATA_DIR = Path('/content/ntl_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / 'train.csv').exists():
    os.system('kaggle competitions download -c name-that-llm -f train.csv -p /content/ntl_data')
    if (DATA_DIR / 'train.csv.zip').exists():
        os.system('unzip -o -q /content/ntl_data/train.csv.zip -d /content/ntl_data')
    assert (DATA_DIR / 'train.csv').exists(), "Failed to download train.csv!"

SPLIT_URL = "https://raw.githubusercontent.com/chiba244qq/name-that-llm-dashboard/main/data/official_5fold_splits.json"
SPLIT_PATH = DATA_DIR / 'official_5fold_splits.json'
urllib.request.urlretrieve(SPLIT_URL, SPLIT_PATH)
split_meta = json.loads(SPLIT_PATH.read_text())

df_train = pd.read_csv(DATA_DIR / 'train.csv')
assert len(df_train) == 2699, f"Unexpected train rows: {len(df_train)}"
assert np.array_equal(df_train['id'].astype(str).values, np.array(split_meta['ids'], dtype=str))

LABELS = [
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra',
    'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'
]
assert set(df_train['model'].unique()) == set(LABELS)
LABEL_TO_ID = {m: i for i, m in enumerate(LABELS)}
y_all = np.array([LABEL_TO_ID[m] for m in df_train['model']], dtype=np.int64)
folds_all = np.array(split_meta['folds'], dtype=np.int64)

for f in range(5):
    tr_p = set(df_train.loc[folds_all != f, 'prompt_id'])
    va_p = set(df_train.loc[folds_all == f, 'prompt_id'])
    assert len(tr_p & va_p) == 0, f"Prompt leakage in fold {f}!"

# --- Open Pool (Train Fold Injection Only) ---
USE_EXTERNAL = True
MAX_EXT_PER_CLASS = 500   # Balanced cap (~9,000 rows max) to keep A100 runtime ~12-15 min/fold
df_ext = pd.DataFrame(columns=['text', 'y'])

if USE_EXTERNAL:
    pool_dir = DATA_DIR / 'open_pool'
    pool_dir.mkdir(exist_ok=True)
    if not list(pool_dir.glob('*.parquet')):
        os.system(f'kaggle datasets download -d crtproduction/name-that-llm-open-pool -p {pool_dir} --unzip')
    pq_files = sorted(pool_dir.glob('*.parquet'))
    assert pq_files, "Open Pool parquet not found!"
    pool = pd.read_parquet(pq_files[0])
    print(f"Raw open_pool rows={len(pool)}, cols={list(pool.columns)}")
    if 'flag_train_prompt_regen' in pool.columns:
        pool = pool[~pool['flag_train_prompt_regen'].fillna(False)]
    pool = pool[pool['model'].isin(LABELS)].dropna(subset=['text', 'model']).copy()
    train_texts_set = set(df_train['text'].astype(str).str.strip())
    pool = pool[~pool['text'].astype(str).str.strip().isin(train_texts_set)].copy()
    pool['y'] = pool['model'].map(LABEL_TO_ID).astype(np.int64)
    if 'match_weight' in pool.columns:
        pool = pool.sort_values('match_weight', ascending=False)
        df_ext = pool.groupby('y', group_keys=False).head(MAX_EXT_PER_CLASS)[['text', 'y']].reset_index(drop=True)
    else:
        df_ext = pool.groupby('y', group_keys=False).apply(
            lambda g: g.sample(n=min(MAX_EXT_PER_CLASS, len(g)), random_state=42)
        )[['text', 'y']].reset_index(drop=True)
    print(f"Filtered Open Pool rows for Train injection: {len(df_ext)} (per-class min/max = {df_ext['y'].value_counts().min()}/{df_ext['y'].value_counts().max()})")


In [ ]:
# [Cell 3] Gemma-4-12B Multi-Depth 18-Class LoRA Architecture & Config
MODEL_ID = "mistralai/Mistral-7B-v0.3"  # Alternative: "google/gemma-2-9b"
MAX_LEN = 384                     # Matches NTL 0707 infer.py max_len=384
BATCH_SIZE = 8
GRAD_ACCUM = 2                    # Effective batch size = 16 (7B L4 fast)
EPOCHS = 2                    # Peaked at E2 in Fold 0; ~57 min/fold on L4
LR_LORA = 1.0e-4
LR_HEAD = 3.0e-4
LORA_R = 16
LORA_ALPHA = 32
LABEL_SMOOTHING = 0.05
N_FOLDS_TO_RUN = 5                # Set to 1 for ~15 min Fold-0 quick check, 5 for full 2699 OOF
SAVE_ADAPTERS = True

USE_4BIT = (VRAM_GB > 0 and VRAM_GB < 45.0)
print(f"Config -> MODEL={MODEL_ID} | MAX_LEN={MAX_LEN} | USE_4BIT={USE_4BIT} | FOLDS={N_FOLDS_TO_RUN}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class Text18Dataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [str(t) for t in texts]
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], int(self.labels[idx])

def collate_fn(batch):
    txts, lbls = zip(*batch)
    enc = tokenizer(list(txts), padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    enc['labels'] = torch.tensor(lbls, dtype=torch.long)
    return enc

class MultiDepth18ClassLoRA(nn.Module):
    def __init__(self, model_id=MODEL_ID, use_4bit=USE_4BIT):
        super().__init__()
        dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
        kwargs = {"dtype": dtype, "attn_implementation": "sdpa"}
        if use_4bit:
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype
            )
            kwargs["device_map"] = {"": torch.cuda.current_device()}

        backbone = AutoModel.from_pretrained(model_id, **kwargs)
        if hasattr(backbone, "language_model"):
            backbone = backbone.language_model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        backbone.config.use_cache = False
        if use_4bit:
            backbone = prepare_model_for_kbit_training(backbone, use_gradient_checkpointing=True)
        else:
            backbone.enable_input_require_grads()
            try:
                backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except TypeError:
                backbone.gradient_checkpointing_enable()

        lora_cfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none"
        )
        self.backbone = get_peft_model(backbone, lora_cfg)
        if torch.cuda.is_available():
            _dev = torch.device(f"cuda:{torch.cuda.current_device()}")
            for _m in self.backbone.modules():
                if any(p.device.type == "cpu" for p in _m.parameters(recurse=False)) or any(b.device.type == "cpu" for b in _m.buffers(recurse=False)):
                    _m.to(_dev)
        text_cfg = getattr(backbone.config, "text_config", backbone.config)
        n_layers = int(text_cfg.num_hidden_layers)
        hdim = int(text_cfg.hidden_size)
        self.tap_layers = sorted(set([
            max(1, int(round(n_layers * 0.30))),
            max(1, int(round(n_layers * 0.55))),
            max(1, int(round(n_layers * 0.80))),
            n_layers,
        ]))
        n_taps = len(self.tap_layers)
        self.mean_norms = nn.ModuleList([nn.LayerNorm(hdim) for _ in range(n_taps)])
        self.last_norms = nn.ModuleList([nn.LayerNorm(hdim) for _ in range(n_taps)])
        self.mean_gate = nn.Parameter(torch.zeros(n_taps))
        self.last_gate = nn.Parameter(torch.zeros(n_taps))
        self.dropout = nn.Dropout(0.15)
        self.head = nn.Linear(hdim * 2, len(LABELS))
        print(f"Initialized {model_id} | layers={n_layers} | taps={self.tap_layers} | hdim={hdim} | 4bit={use_4bit}")

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True, use_cache=False)
        hs = out.hidden_states if getattr(out, "hidden_states", None) is not None else out.language_model_outputs.hidden_states
        mask = attention_mask.unsqueeze(-1).float()
        denom = mask.sum(dim=1).clamp(min=1e-6)

        w_mean = torch.softmax(self.mean_gate, dim=0)
        w_last = torch.softmax(self.last_gate, dim=0)

        mean_feats, last_feats = [], []
        for idx, layer_id in enumerate(self.tap_layers):
            h = hs[layer_id].float()
            v_mean = (h * mask).sum(dim=1) / denom
            v_last = h[:, -1, :]
            mean_feats.append(w_mean[idx] * self.mean_norms[idx](v_mean))
            last_feats.append(w_last[idx] * self.last_norms[idx](v_last))

        feat = torch.cat([sum(mean_feats), sum(last_feats)], dim=-1)
        return self.head(self.dropout(feat))


In [ ]:
# [Cell 4] Leak-Free 5-Fold GroupKFold Training & OOF Generation
C3_IDX = [LABELS.index('glm-5.3'), LABELS.index('glm-5.3-flash'), LABELS.index('claude-fable-5')]
GEM_IDX = [LABELS.index('gemini-3.7-flash'), LABELS.index('gemini-3.8-flash')]
SOL_IDX = [LABELS.index('gpt-5.6-luna'), LABELS.index('gpt-5.6-sol'), LABELS.index('gpt-5.6-terra')]

def sync_to_github(fold_done):
    gh_token = os.environ.get('GH_TOKEN', '')
    if not gh_token:
        print(f"  [GitHub Sync] Skipped after Fold {fold_done} (no GITHUB_TOKEN).")
        return
    try:
        repo_dir = Path('/content/name-that-llm-dashboard')
        if not repo_dir.exists():
            subprocess.run(
                ['git', 'clone', '-q', f'https://{gh_token}@github.com/chiba244qq/name-that-llm-dashboard.git', str(repo_dir)],
                check=True,
            )
            subprocess.run(['git', '-C', str(repo_dir), 'config', 'user.name', 'colab-runner'], check=True)
            subprocess.run(['git', '-C', str(repo_dir), 'config', 'user.email', 'colab@local'], check=True)
        else:
            subprocess.run(['git', '-C', str(repo_dir), 'pull', '--rebase', '-q'], check=False)
        res_dir = repo_dir / 'data' / 'colab_results'
        res_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy('/content/phase9_mistral7b_oof_2699.npy', res_dir / 'phase9_mistral7b_oof_2699.npy')
        shutil.copy('/content/phase9_mistral7b_summary.json', res_dir / 'phase9_mistral7b_summary.json')
        subprocess.run(['git', '-C', str(repo_dir), 'add', 'data/colab_results/phase9_mistral7b_oof_2699.npy', 'data/colab_results/phase9_mistral7b_summary.json'], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'commit', '-m', f'Phase 9 Mistral-7B OOF checkpoint (through Fold {fold_done})'], check=False)
        subprocess.run(['git', '-C', str(repo_dir), 'push', '-q'], check=True)
        print(f"  [GitHub Sync] Pushed Fold {fold_done} checkpoint to origin/main!", flush=True)
    except Exception as e:
        print(f"  [GitHub Sync] Warning (Fold {fold_done}): {e}", flush=True)

def subset_acc(y_true, logits, idx_list):
    m = np.isin(y_true, idx_list)
    yt = np.array([idx_list.index(int(y)) for y in y_true[m]])
    yp = logits[m][:, idx_list].argmax(1)
    return float(accuracy_score(yt, yp))

oof_logits = np.zeros((len(df_train), len(LABELS)), dtype=np.float32)
fold_summaries = {}
adapter_root = Path('/content/phase9_adapters')
adapter_root.mkdir(parents=True, exist_ok=True)
texts_all = df_train['text'].tolist()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

for fold in range(N_FOLDS_TO_RUN):
    t0 = time.time()
    torch.manual_seed(42 + fold)
    np.random.seed(42 + fold)

    tr_idx = np.where(folds_all != fold)[0]
    va_idx = np.where(folds_all == fold)[0]

    tr_texts = [texts_all[i] for i in tr_idx]
    tr_y = list(y_all[tr_idx])
    if USE_EXTERNAL and len(df_ext) > 0:
        tr_texts.extend(df_ext['text'].astype(str).tolist())
        tr_y.extend(df_ext['y'].astype(int).tolist())

    tr_ds = Text18Dataset(tr_texts, tr_y)
    va_ds = Text18Dataset([texts_all[i] for i in va_idx], y_all[va_idx])

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

    model = MultiDepth18ClassLoRA(MODEL_ID, use_4bit=USE_4BIT)
    if not USE_4BIT:
        model.to(device)
    else:
        for mod in [model.mean_norms, model.last_norms, model.dropout, model.head]:
            mod.to(device)
        model.mean_gate.data = model.mean_gate.data.to(device)
        model.last_gate.data = model.last_gate.data.to(device)

    lora_params = [p for _, p in model.backbone.named_parameters() if p.requires_grad]
    head_params = (
        list(model.mean_norms.parameters()) + list(model.last_norms.parameters())
        + list(model.head.parameters()) + [model.mean_gate, model.last_gate]
    )
    optimizer = torch.optim.AdamW([
        {"params": lora_params, "lr": LR_LORA, "weight_decay": 0.01},
        {"params": head_params, "lr": LR_HEAD, "weight_decay": 0.02},
    ])
    total_steps = max(1, (len(tr_loader) + GRAD_ACCUM - 1) // GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.10 * total_steps), num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    best_bacc, best_logits, best_epoch = -1.0, None, 0
    print(f"\n========== Fold {fold} (Train={len(tr_ds)} [Official={len(tr_idx)} + Ext={len(tr_ds)-len(tr_idx)}], Val={len(va_idx)}) ==========")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        tr_loss = 0.0
        t_ep = time.time()
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(tr_loader):
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            with torch.amp.autocast('cuda', dtype=amp_dtype):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits.float(), labels) / GRAD_ACCUM
            loss.backward()
            tr_loss += float(loss.item()) * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(tr_loader):
                torch.nn.utils.clip_grad_norm_(lora_params + head_params, 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            if (step + 1) % 50 == 0 or (step + 1) == len(tr_loader):
                elapsed = time.time() - t_ep
                print(f"    [F{fold} E{epoch}] step {step+1}/{len(tr_loader)} | avg_loss={tr_loss/(step+1):.4f} | {elapsed:.0f}s", flush=True)

        model.eval()
        val_logits_list = []
        with torch.no_grad():
            for batch in va_loader:
                input_ids = batch['input_ids'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                with torch.amp.autocast('cuda', dtype=amp_dtype):
                    logits = model(input_ids, attention_mask)
                val_logits_list.append(logits.float().cpu().numpy())

        val_logits = np.vstack(val_logits_list)
        val_preds = val_logits.argmax(1)
        val_bacc = float(balanced_accuracy_score(y_all[va_idx], val_preds))
        c3_acc = subset_acc(y_all[va_idx], val_logits, C3_IDX)
        gem_acc = subset_acc(y_all[va_idx], val_logits, GEM_IDX)
        print(f"  Epoch {epoch}/{EPOCHS} | loss={tr_loss/len(tr_loader):.4f} | 18-cls BAcc={val_bacc:.4f} | GLM/Fable5={c3_acc:.4f} | Gem3.7/3.8={gem_acc:.4f} | uniq={len(np.unique(val_preds))}/18")
        if val_bacc > best_bacc:
            best_bacc, best_logits, best_epoch = val_bacc, val_logits.copy(), epoch
            if SAVE_ADAPTERS:
                fdir = adapter_root / f"fold{fold}"
                fdir.mkdir(parents=True, exist_ok=True)
                model.backbone.save_pretrained(fdir)
                torch.save({
                    "tap_layers": model.tap_layers,
                    "mean_norms": model.mean_norms.state_dict(),
                    "last_norms": model.last_norms.state_dict(),
                    "mean_gate": model.mean_gate.detach().cpu(),
                    "last_gate": model.last_gate.detach().cpu(),
                    "head": model.head.state_dict(),
                    "labels": LABELS,
                }, fdir / "head.pt")

    oof_logits[va_idx] = best_logits.astype(np.float32)
    np.save('/content/phase9_mistral7b_oof_2699.npy', oof_logits)
    fold_summaries[f"fold{fold}"] = {
        "best_epoch": best_epoch,
        "best_bacc": round(best_bacc, 5),
        "glm_fable5_acc": round(subset_acc(y_all[va_idx], best_logits, C3_IDX), 4),
        "gem37_38_acc": round(subset_acc(y_all[va_idx], best_logits, GEM_IDX), 4),
        "elapsed_sec": round(time.time() - t0, 1),
    }
    _done_idx = np.where(folds_all <= fold)[0]
    _cur_summary = {
        "model_id": MODEL_ID,
        "use_external": bool(USE_EXTERNAL),
        "ext_rows": int(len(df_ext)),
        "max_len": MAX_LEN,
        "epochs": EPOCHS,
        "folds_completed": fold + 1,
        "rows_evaluated": int(len(_done_idx)),
        "overall_18class_bacc": round(float(balanced_accuracy_score(y_all[_done_idx], oof_logits[_done_idx].argmax(1))), 5),
        "folds": fold_summaries,
    }
    Path('/content/phase9_mistral7b_summary.json').write_text(json.dumps(_cur_summary, indent=2))
    print(f"Fold {fold} done -> Best Epoch {best_epoch} | BAcc={best_bacc:.5f} | Time={fold_summaries[f'fold{fold}']['elapsed_sec']}s", flush=True)
    sync_to_github(fold)
    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

pop_idx = np.where(folds_all < N_FOLDS_TO_RUN)[0]
overall_bacc = float(balanced_accuracy_score(y_all[pop_idx], oof_logits[pop_idx].argmax(1)))
np.save('/content/phase9_mistral7b_oof_2699.npy', oof_logits)
summary = {
    "model_id": MODEL_ID,
    "use_external": bool(USE_EXTERNAL),
    "ext_rows": int(len(df_ext)),
    "max_len": MAX_LEN,
    "epochs": EPOCHS,
    "n_folds_run": N_FOLDS_TO_RUN,
    "rows_evaluated": int(len(pop_idx)),
    "overall_18class_bacc": round(overall_bacc, 5),
    "glm_fable5_3class_acc": round(subset_acc(y_all[pop_idx], oof_logits[pop_idx], C3_IDX), 4),
    "gem37_38_2class_acc": round(subset_acc(y_all[pop_idx], oof_logits[pop_idx], GEM_IDX), 4),
    "gpt56_3class_acc": round(subset_acc(y_all[pop_idx], oof_logits[pop_idx], SOL_IDX), 4),
    "folds": fold_summaries,
}
Path('/content/phase9_mistral7b_summary.json').write_text(json.dumps(summary, indent=2))
print("\n=== PHASE 9 OOF SUMMARY ===")
print(json.dumps(summary, indent=2))


In [ ]:
# [Cell 5] Automated GitHub Result Sync (phase9 OOF .npy + summary -> Dashboard Repo)
gh_token = ""
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN') or ""
except Exception:
    pass
if not gh_token:
    gh_token = os.environ.get('GH_TOKEN', '')
if not gh_token:
    gh_token = getpass.getpass("GitHub Token (leave blank to skip push): ").strip()

if gh_token:
    repo_dir = Path('/content/name-that-llm-dashboard')
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    os.system(f"git clone -q https://{gh_token}@github.com/chiba244qq/name-that-llm-dashboard.git {repo_dir}")
    res_dir = repo_dir / 'data' / 'colab_results'
    res_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy('/content/phase9_mistral7b_oof_2699.npy', res_dir / 'phase9_mistral7b_oof_2699.npy')
    shutil.copy('/content/phase9_mistral7b_summary.json', res_dir / 'phase9_mistral7b_summary.json')
    os.system(
        f'cd {repo_dir} && git config user.name "colab-runner" && git config user.email "colab@local" '
        '&& git add data/colab_results/phase9_mistral7b_oof_2699.npy data/colab_results/phase9_mistral7b_summary.json '
        '&& git commit -m "Add Phase 9 Mistral-7B-v0.3 Open Pool OOF results" && git push -q'
    )
    print("Pushed Phase 9 OOF results to GitHub!")
else:
    print("Skipped GitHub push. Download /content/phase9_mistral7b_oof_2699.npy manually.")
